# Research Companion v0 — Qwen 4B + Unsloth LoRA Fine-Tuning

This Colab notebook fine-tunes a small Qwen model to behave like a **citation-first research chatbot**.

Goal:

```text
User asks factual/current/research question
→ model decides search is needed
→ model emits a structured tool call
→ backend/search tool returns sources
→ model reads sources
→ model answers with citations and uncertainty
```

This notebook is for **v0 training**, not final production.  
Start small, verify it trains, then scale dataset size and steps.

## 0. Colab runtime setup

In Colab:

```text
Runtime → Change runtime type → Hardware accelerator → T4 GPU
```

Then run the cells top to bottom.

Notes:

- Start with `MAX_SEQ_LENGTH = 2048`.
- Do not attempt 128k/1M context during fine-tuning.
- Qwen3.5 can compile custom kernels slowly on T4. The first training run may feel stuck while compiling.
- For Qwen3.5, Unsloth docs do **not** recommend 4-bit QLoRA because of larger quantization differences, so this notebook starts with 16-bit LoRA.

In [2]:
# Check GPU
!nvidia-smi

Mon Jun 29 11:51:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Install dependencies

This may take a few minutes.  
If Colab asks you to restart runtime after install, restart and continue from the next cell.

In [1]:
%%capture
%pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
%pip install -U datasets trl accelerate bitsandbytes sentencepiece protobuf huggingface_hub pandas

In [2]:
import os, json, random, textwrap, torch
from datasets import Dataset, load_dataset
from huggingface_hub import notebook_login

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Optional Hugging Face login

Run this if you want to push the LoRA adapter or GGUF later.

In [ ]:
# Optional. Run only if you want to push results to Hugging Face.
# notebook_login()

## 3. Training config

Main model target: `unsloth/Qwen3.5-4B`.

Fallback: `unsloth/Qwen3-4B-Instruct-2507`.

Use the fallback if Qwen3.5 has install/kernel issues in Colab.

In [3]:
BASE_MODEL = "unsloth/Qwen3.5-4B"

# Fallback if Qwen3.5 fails on your Colab runtime:
# BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"

OUTPUT_DIR = "research-companion-qwen4b-lora"

# Replace before pushing:
HF_ADAPTER_REPO = "YOUR_HF_USERNAME/research-companion-qwen4b-lora"
HF_GGUF_REPO = "YOUR_HF_USERNAME/research-companion-qwen4b-gguf"

MAX_SEQ_LENGTH = 2048
MAX_STEPS = 80                 # v0 smoke test. Later try 300, 1000, etc.
TRAIN_HOTPOT_SAMPLES = 500     # v0. Later try 5k-50k if runtime allows.
EVAL_SIZE = 0.05
SEED = 3407

# Qwen3.5 docs recommend 16-bit LoRA instead of 4-bit QLoRA.
LOAD_IN_4BIT = False
LOAD_IN_16BIT = True

print("Base model:", BASE_MODEL)

Base model: unsloth/Qwen3.5-4B


## 4. Load model + LoRA adapter

This uses Unsloth's `FastLanguageModel`.

For T4, keep sequence length small first. If you hit OOM, reduce:

```python
MAX_SEQ_LENGTH = 1024
MAX_STEPS = 30
TRAIN_HOTPOT_SAMPLES = 100
```

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    load_in_16bit = LOAD_IN_16BIT,
    fast_inference = False,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    max_seq_length = MAX_SEQ_LENGTH,
)

print("Model and LoRA adapter loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Model and LoRA adapter loaded.


## 5. Research-chatbot behavior spec

We train the model to follow this contract:

```text
- If the question is factual/current/research-heavy, emit a tool call.
- Search tool returns source blocks.
- Final answer must cite source IDs like [S1], [S2].
- If evidence is weak, say so.
- Do not invent citations.
```

The companion app backend can later parse `<tool_call>{...}</tool_call>`.

In [5]:
SYSTEM_PROMPT = '''You are Research Companion, a citation-first research chatbot.

Rules:
1. For factual, current, benchmark, product, legal, medical, financial, or research questions, request search/tool use before answering.
2. Use structured tool calls in this exact XML-like wrapper:
   <tool_call>{"name":"web_search","arguments":{"query":"..."}}</tool_call>
3. After sources are provided, answer only from the sources.
4. Cite source IDs inline as [S1], [S2].
5. If sources disagree, explain the disagreement.
6. If the evidence is insufficient, say "I do not have enough evidence" and explain what is missing.
7. Be concise, accurate, and avoid unsupported claims.
'''

def chat_text(messages):
    '''Apply the model chat template when available.'''
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        # fallback plain format
        out = []
        for m in messages:
            out.append(f"{m['role'].upper()}: {m['content']}")
        return "\n\n".join(out)

def make_source_block(sources):
    lines = []
    for s in sources:
        lines.append(f"[{s['id']}] {s['title']}\nURL: {s.get('url','local-dataset')}\n{s['text']}")
    return "\n\n".join(lines)

def make_tool_call(query):
    return '<tool_call>' + json.dumps({
        "name": "web_search",
        "arguments": {"query": query}
    }, ensure_ascii=False) + '</tool_call>'

## 6. Seed tool-use examples

These are small handcrafted examples that teach the exact app protocol.

The premade dataset comes next.

In [6]:
seed_examples = [
    {
        "user": "Is a local 4B model enough to beat GPT-class models at research?",
        "query": "small language models search tools research assistant accuracy",
        "sources": [
            {
                "id": "S1",
                "title": "Research-agent design note",
                "url": "local-seed://research-agent",
                "text": "Small models can perform well in narrow workflows when paired with retrieval, tools, verification, and citations. They are weaker than frontier models as raw general reasoning systems."
            },
            {
                "id": "S2",
                "title": "RAG reliability note",
                "url": "local-seed://rag",
                "text": "Retrieval-augmented systems reduce hallucination risk when final answers are constrained to retrieved evidence and uncertainty is stated when evidence is insufficient."
            }
        ],
        "answer": "A 4B model is unlikely to beat frontier models as a raw general model. It can still be useful as a research system if it searches, verifies, and answers only from retrieved evidence [S1][S2]."
    },
    {
        "user": "Should I answer current benchmark questions from memory?",
        "query": "latest AI benchmark leaderboard current model scores",
        "sources": [
            {
                "id": "S1",
                "title": "Benchmark freshness note",
                "url": "local-seed://benchmarks",
                "text": "Current model benchmark results change frequently. Answers about latest scores should be verified against current leaderboards or official model pages."
            }
        ],
        "answer": "No. Current benchmark questions should trigger search first because leaderboard positions and model scores can change quickly [S1]."
    },
    {
        "user": "What should the assistant do if sources conflict?",
        "query": "source conflict research answer handling",
        "sources": [
            {
                "id": "S1",
                "title": "Source conflict policy",
                "url": "local-seed://conflict-policy",
                "text": "When sources conflict, a research assistant should state the disagreement, identify which sources support each claim, and avoid presenting either side as settled."
            }
        ],
        "answer": "It should state that the sources conflict, cite which source supports each claim, and avoid treating the issue as settled unless stronger evidence is available [S1]."
    },
]

def seed_to_rows(examples):
    rows = []
    for ex in examples:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex["user"]},
            {"role": "assistant", "content": make_tool_call(ex["query"])},
            {"role": "tool", "content": make_source_block(ex["sources"])},
            {"role": "assistant", "content": ex["answer"]},
        ]
        rows.append({"text": chat_text(messages), "source": "seed_tool_use"})
    return rows

rows = seed_to_rows(seed_examples)
print(rows[0]["text"][:1000])

<|im_start|>system
You are Research Companion, a citation-first research chatbot.

Rules:
1. For factual, current, benchmark, product, legal, medical, financial, or research questions, request search/tool use before answering.
2. Use structured tool calls in this exact XML-like wrapper:
   <tool_call>{"name":"web_search","arguments":{"query":"..."}}</tool_call>
3. After sources are provided, answer only from the sources.
4. Cite source IDs inline as [S1], [S2].
5. If sources disagree, explain the disagreement.
6. If the evidence is insufficient, say "I do not have enough evidence" and explain what is missing.
7. Be concise, accurate, and avoid unsupported claims.<|im_end|>
<|im_start|>user
Is a local 4B model enough to beat GPT-class models at research?<|im_end|>
<|im_start|>assistant
<think>

</think>

<tool_call>{"name": "web_search", "arguments": {"query": "small language models search tools research assistant accuracy"}}</tool_call><|im_end|>
<|im_start|>user
<tool_response>
[S1] R

## 7. Load premade research QA data: HotpotQA

HotpotQA is useful because it contains multi-hop QA with supporting Wikipedia context.

We convert examples into:

```text
question → tool call → source blocks → cited final answer
```

This is not perfect yet, but good enough for v0 behavior tuning.

In [7]:
def build_hotpot_rows(n=500):
    hotpot = load_dataset("hotpot_qa", "distractor", split=f"train[:{n}]")
    out = []

    for ex in hotpot:
        question = ex["question"]
        answer = ex["answer"]

        titles = ex["context"]["title"]
        sentences_list = ex["context"]["sentences"]

        sources = []
        for i, (title, sentences) in enumerate(zip(titles[:6], sentences_list[:6])):
            text = " ".join(sentences[:4])
            text = text.replace("\n", " ").strip()
            if not text:
                continue
            sources.append({
                "id": f"S{i+1}",
                "title": title,
                "url": "hotpotqa://wikipedia-context",
                "text": text[:1200],
            })

        if not sources:
            continue

        # Use supporting fact titles for citation IDs when possible.
        citation_ids = []
        try:
            supporting_titles = set(ex["supporting_facts"]["title"])
            for s in sources:
                if s["title"] in supporting_titles:
                    citation_ids.append(f"[{s['id']}]")
        except Exception:
            pass

        if not citation_ids:
            citation_ids = ["[S1]"]

        cited_answer = (
            f"Based on the provided sources, the answer is: {answer}. "
            f"Evidence: {''.join(citation_ids[:3])}"
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
            {"role": "assistant", "content": make_tool_call(question)},
            {"role": "tool", "content": make_source_block(sources)},
            {"role": "assistant", "content": cited_answer},
        ]

        out.append({"text": chat_text(messages), "source": "hotpot_qa"})

    return out

try:
    hotpot_rows = build_hotpot_rows(TRAIN_HOTPOT_SAMPLES)
    rows.extend(hotpot_rows)
    print("Loaded HotpotQA rows:", len(hotpot_rows))
except Exception as e:
    print("HotpotQA load failed. Continuing with seed examples only.")
    print(type(e).__name__, str(e)[:500])

print("Total rows:", len(rows))

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

HotpotQA load failed. Continuing with seed examples only.
HfUriError Invalid HF URI 'hf://datasets/hotpot_qa@1908d6afbbead072334abe2965f91bd2709910ab/.huggingface.yaml'. Repository id must be 'namespace/name', got 'hotpot_qa'.
Total rows: 3


## 8. Optional: add FEVER-style verification rows

This is optional because Hugging Face dataset configs can change.

If it loads, it adds examples where the model learns:

```text
supported / refuted / not enough evidence
```

In [8]:
ADD_FEVER = False  # Set True later after the v0 notebook trains once.

def build_fever_rows(n=300):
    # Dataset names/configs can vary. This block is intentionally wrapped.
    fever = load_dataset("fever", "v1.0", split=f"train[:{n}]")
    out = []
    for ex in fever:
        claim = str(ex.get("claim", "")).strip()
        label = str(ex.get("label", "")).strip()
        if not claim or not label:
            continue

        sources = [{
            "id": "S1",
            "title": "Claim verification record",
            "url": "fever://claim",
            "text": f"Claim: {claim}\nGold label: {label}"
        }]

        answer = (
            f"The claim verification label is {label}. "
            f"This should be treated as a verification task, not an unsupported factual answer [S1]."
        )

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Verify this claim: {claim}"},
            {"role": "assistant", "content": make_tool_call(claim)},
            {"role": "tool", "content": make_source_block(sources)},
            {"role": "assistant", "content": answer},
        ]
        out.append({"text": chat_text(messages), "source": "fever"})
    return out

if ADD_FEVER:
    try:
        fever_rows = build_fever_rows(300)
        rows.extend(fever_rows)
        print("Loaded FEVER rows:", len(fever_rows))
    except Exception as e:
        print("FEVER load failed. Skipping.")
        print(type(e).__name__, str(e)[:500])

print("Total rows:", len(rows))

Total rows: 3


## 9. Create train/eval dataset

We use a simple `text` column because TRL SFT supports standard text datasets.

In [9]:
random.seed(SEED)
random.shuffle(rows)

dataset = Dataset.from_list(rows)
dataset = dataset.shuffle(seed=SEED)

if len(dataset) > 20:
    split = dataset.train_test_split(test_size=EVAL_SIZE, seed=SEED)
    train_dataset = split["train"]
    eval_dataset = split["test"]
else:
    train_dataset = dataset
    eval_dataset = None

print(train_dataset)
print("Eval:", eval_dataset)
print("\nPreview:\n")
print(train_dataset[0]["text"][:2000])

Dataset({
    features: ['text', 'source'],
    num_rows: 3
})
Eval: None

Preview:

<|im_start|>system
You are Research Companion, a citation-first research chatbot.

Rules:
1. For factual, current, benchmark, product, legal, medical, financial, or research questions, request search/tool use before answering.
2. Use structured tool calls in this exact XML-like wrapper:
   <tool_call>{"name":"web_search","arguments":{"query":"..."}}</tool_call>
3. After sources are provided, answer only from the sources.
4. Cite source IDs inline as [S1], [S2].
5. If sources disagree, explain the disagreement.
6. If the evidence is insufficient, say "I do not have enough evidence" and explain what is missing.
7. Be concise, accurate, and avoid unsupported claims.<|im_end|>
<|im_start|>user
Should I answer current benchmark questions from memory?<|im_end|>
<|im_start|>assistant
<think>

</think>

<tool_call>{"name": "web_search", "arguments": {"query": "latest AI benchmark leaderboard current model scor

## 10. Train

This is a smoke-test run.  
For a real v1, increase:

```python
MAX_STEPS = 300 to 1000+
TRAIN_HOTPOT_SAMPLES = 5000+
```

Do not scale until this small run finishes successfully.

In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        logging_steps = 1,
        output_dir = "outputs_research_companion",
        optim = "adamw_8bit",
        seed = SEED,
        dataset_num_proc = 1,
        fp16 = True,
        bf16 = False,
        report_to = "none",
    ),
)

trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/3 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 80 | Total steps = 80
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Step,Training Loss
1,2.785882
2,2.785882
3,2.733629
4,2.552798
5,2.344371
6,2.170247
7,2.002977
8,1.806182
9,1.581761
10,1.339301


Unsloth: Restored added_tokens_decoder metadata in outputs_research_companion/checkpoint-80/tokenizer_config.json.


TrainOutput(global_step=80, training_loss=0.335270767021575, metrics={'train_runtime': 579.7735, 'train_samples_per_second': 0.552, 'train_steps_per_second': 0.138, 'total_flos': 1790515732561920.0, 'train_loss': 0.335270767021575, 'epoch': 80.0})

## 11. Save LoRA adapter locally

This saves only the LoRA adapter, not a full merged model.

In [11]:
import os, json, shutil
from datetime import datetime

RUN_NAME = f"kenprobe-qwen35-4b-lora-step{MAX_STEPS}"

ADAPTER_DIR = f"adapters/{RUN_NAME}"
REPORT_DIR = "reports"
RUN_DIR = f"runs/step{MAX_STEPS}"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

report = {
    "run_name": RUN_NAME,
    "created_at": datetime.utcnow().isoformat() + "Z",
    "base_model": BASE_MODEL,
    "max_steps": MAX_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_hotpot_samples": TRAIN_HOTPOT_SAMPLES,
    "eval_size": EVAL_SIZE,
    "load_in_4bit": LOAD_IN_4BIT,
    "load_in_16bit": LOAD_IN_16BIT,
    "adapter_dir": ADAPTER_DIR,
}

REPORT_PATH = f"{REPORT_DIR}/{RUN_NAME}.json"

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Saved adapter:", ADAPTER_DIR)
print("Saved report:", REPORT_PATH)

!ls -lh {ADAPTER_DIR}
!cat {REPORT_PATH}

Unsloth: Restored added_tokens_decoder metadata in adapters/kenprobe-qwen35-4b-lora-step80/tokenizer_config.json.


Saved adapter: adapters/kenprobe-qwen35-4b-lora-step80
Saved report: reports/kenprobe-qwen35-4b-lora-step80.json
total 60M
-rw-r--r-- 1 root root 1.5K Jun 29 12:25 adapter_config.json
-rw------- 1 root root  41M Jun 29 12:25 adapter_model.safetensors
-rw-r--r-- 1 root root 7.9K Jun 29 12:25 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Jun 29 12:25 processor_config.json
-rw-r--r-- 1 root root 5.1K Jun 29 12:25 README.md
-rw-r--r-- 1 root root 7.0K Jun 29 12:25 tokenizer_config.json
-rw-r--r-- 1 root root  20M Jun 29 12:25 tokenizer.json


/tmp/ipykernel_7023/1950793589.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",


{
  "run_name": "kenprobe-qwen35-4b-lora-step80",
  "created_at": "2026-06-29T12:25:16.992753Z",
  "base_model": "unsloth/Qwen3.5-4B",
  "max_steps": 80,
  "max_seq_length": 2048,
  "train_hotpot_samples": 500,
  "eval_size": 0.05,
  "load_in_4bit": false,
  "load_in_16bit": true,
  "adapter_dir": "adapters/kenprobe-qwen35-4b-lora-step80"
}

In [12]:
import os, shutil

RUN_NAME = f"kenprobe-qwen35-4b-lora-step{MAX_STEPS}"
ADAPTER_DIR = f"adapters/{RUN_NAME}"
REPORT_PATH = f"reports/{RUN_NAME}.json"

BUNDLE_DIR = f"runs/{RUN_NAME}"
os.makedirs(BUNDLE_DIR, exist_ok=True)

shutil.copytree(ADAPTER_DIR, f"{BUNDLE_DIR}/adapter", dirs_exist_ok=True)
shutil.copy(REPORT_PATH, f"{BUNDLE_DIR}/report.json")

BUNDLE_ZIP = shutil.make_archive(BUNDLE_DIR, "zip", BUNDLE_DIR)

print("Bundle created:", BUNDLE_ZIP)
!ls -lh {BUNDLE_ZIP}

Bundle created: /content/runs/kenprobe-qwen35-4b-lora-step80.zip
-rw-r--r-- 1 root root 36M Jun 29 12:31 /content/runs/kenprobe-qwen35-4b-lora-step80.zip


In [23]:
from google.colab import files

files.download("/content/runs/kenprobe-qwen35-4b-lora-step80.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
!ls -lh /content/runs/

total 36M
drwxr-xr-x 3 root root 4.0K Jun 29 12:31 kenprobe-qwen35-4b-lora-step80
-rw-r--r-- 1 root root  36M Jun 29 12:31 kenprobe-qwen35-4b-lora-step80.zip
drwxr-xr-x 2 root root 4.0K Jun 29 12:25 step80


In [20]:
from IPython.display import FileLink, display

display(FileLink("/content/runs/kenprobe-qwen35-4b-lora-step80.zip"))

/content/runs/kenprobe-qwen35-4b-lora-step80.zip

In [24]:
import os, subprocess

ZIP_PATH = "/content/runs/kenprobe-qwen35-4b-lora-step80.zip"
assert os.path.exists(ZIP_PATH), f"Missing: {ZIP_PATH}"

result = subprocess.check_output(
    ["curl", "-F", f"file=@{ZIP_PATH}", "https://0x0.st"],
    text=True
)

print("Download URL:")
print(result.strip())

Download URL:
uploads disabled because it’s been almost nothing but AI botnet spam for the past few months. will be back with a few changes at some point. no ETA.


In [25]:
from huggingface_hub import login, HfApi
import os

login()  # paste your Hugging Face write token when asked

HF_REPO = "amaansyed27/kenprobe-adapters"  # change if you want another repo name
ZIP_PATH = "/content/runs/kenprobe-qwen35-4b-lora-step80.zip"

api = HfApi()
api.create_repo(
    repo_id=HF_REPO,
    repo_type="model",
    private=True,
    exist_ok=True,
)

api.upload_file(
    path_or_fileobj=ZIP_PATH,
    path_in_repo="kenprobe-qwen35-4b-lora-step80.zip",
    repo_id=HF_REPO,
    repo_type="model",
)

print("Uploaded to:", HF_REPO)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...qwen35-4b-lora-step80.zip:   1%|1         |  524kB / 37.2MB            

Uploaded to: amaansyed27/kenprobe-adapters


## 12. Quick inference test

This tests whether the model learned the **tool-call-first** behavior.

In [27]:
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)

# Some Qwen3.5 builds return a processor-like object.
# Use the inner text tokenizer if it exists.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

def to_chatml(messages, add_generation_prompt=True):
    parts = []
    for m in messages:
        role = m["role"]
        content = m["content"]
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    if add_generation_prompt:
        parts.append("<|im_start|>assistant\n")
    return "\n".join(parts)

def generate_text(messages, max_new_tokens=512):
    prompt = to_chatml(messages, add_generation_prompt=True)

    inputs = text_tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            pad_token_id=text_tokenizer.eos_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return text_tokenizer.decode(new_tokens, skip_special_tokens=True)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Compare Qwythos 9B with Gemini Flash for research tasks. Be accurate."},
]

print(generate_text(test_messages, max_new_tokens=350))

<think>

</think>

<tool_call>{"name": "web_search", "arguments": {"query": "Qwythos 9B model research assistant accuracy"}}</tool_call>
<tool_call>{"name": "web_search", "arguments": {"query": "Gemini Flash model research assistant accuracy"}}</tool_call>


## 13. Test with provided sources

This simulates what the companion app will do after executing search.

In [28]:
source_block = """[S1] Qwythos model card
URL: https://huggingface.co/example/qwythos
Qwythos is a 9B model based on Qwen and advertises long-context capability. Its benchmark claims are self-reported on the model card.

[S2] Public benchmark leaderboard
URL: https://example.com/leaderboard
The public leaderboard does not list Qwythos. Frontier cloud models have independent agent benchmark scores that are much higher than small local models.
"""

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Is Qwythos proven to beat Gemini Flash?"},
    {"role": "tool", "content": source_block},
]

print(generate_text(test_messages, max_new_tokens=400))

<think>

</think>

<tool_call>{"name": "web_search", "arguments": {"query": "Qwythos benchmark leaderboard local model vs frontier model"}}</tool_call>


## 14. Push LoRA adapter to Hugging Face

Before running:

```python
HF_ADAPTER_REPO = "your-username/research-companion-qwen4b-lora"
```

Then uncomment and run.

In [ ]:
# Push adapter to Hugging Face.
# Make sure you ran notebook_login() earlier or set HF_TOKEN.

# model.push_to_hub(HF_ADAPTER_REPO)
# tokenizer.push_to_hub(HF_ADAPTER_REPO)

## 15. Optional GGUF export for Ollama / llama.cpp

This can take a long time and may OOM on Colab.

Unsloth supports:

```python
model.save_pretrained_gguf("directory", tokenizer, quantization_method="q4_k_m")
model.push_to_hub_gguf("hf_username/directory", tokenizer, quantization_method="q4_k_m")
```

Run only after the LoRA adapter looks useful.

In [ ]:
DO_GGUF_EXPORT = False

if DO_GGUF_EXPORT:
    GGUF_DIR = "research-companion-qwen4b-Q4_K_M"
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print("Saved GGUF to:", GGUF_DIR)

    # Optional push:
    # model.push_to_hub_gguf(
    #     HF_GGUF_REPO,
    #     tokenizer,
    #     quantization_method = "q4_k_m",
    # )

## 16. Suggested next training upgrades

After this notebook works:

```text
1. Increase HotpotQA samples to 5k+
2. Add ASQA / Natural Questions for long-form answers
3. Add RAGTruth / FEVER for hallucination and verification
4. Add synthetic tool-use examples for:
   - search query generation
   - source ranking
   - citation formatting
   - contradiction handling
5. Build a 100-question eval set before training more
```

Expected v0 behavior:

```text
Good:
- emits search tool calls
- cites source IDs
- admits uncertainty more often
- follows research-answer structure

Not yet good:
- true deep research
- source quality ranking
- multi-round tool loops
- fully reliable factuality
```